## Ноутбук позволяет найти совпдающиеся монеты на фьючерсах OKX и Bybit

Входные данные:---
Выходной результат: текстовые файлы символов на bybit и okx

In [7]:
# fetch_bybit_linear_instruments.py
import csv
import json
import time
from pathlib import Path

import requests

BASE_URL = "https://api.bybit.com"
ENDPOINT = "/v5/market/instruments-info"
OUT_DIR = Path("output/bybit")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def fetch_all_linear_instruments():
    session = requests.Session()
    cursor = None
    all_items = []
    raw_pages = []
    page_num = 0

    while True:
        params = {
            "category": "linear",
            "limit": 1000,
        }
        if cursor:
            params["cursor"] = cursor

        resp = session.get(BASE_URL + ENDPOINT, params=params, timeout=20)
        resp.raise_for_status()
        data = resp.json()

        if data.get("retCode") != 0:
            raise RuntimeError(f"Bybit API error: {data}")

        result = data.get("result", {})
        items = result.get("list", [])
        cursor = result.get("nextPageCursor") or ""

        page_num += 1
        raw_pages.append(data)
        all_items.extend(items)

        print(f"page={page_num} items={len(items)} total={len(all_items)} cursor={'yes' if cursor else 'no'}")

        if not cursor:
            break

        time.sleep(0.05)

    return all_items, raw_pages

def normalize(item):
    price_filter = item.get("priceFilter", {}) or {}
    lot_filter = item.get("lotSizeFilter", {}) or {}

    base = item.get("baseCoin")
    quote = item.get("quoteCoin")
    settle = item.get("settleCoin")

    return {
        "exchange": "bybit",
        "symbol_raw": item.get("symbol"),
        "symbol_norm": base,
        "category": "linear",
        "contract_type": item.get("contractType"),
        "status": item.get("status"),
        "base_coin": base,
        "quote_coin": quote,
        "settle_coin": settle,
        "tick_size": price_filter.get("tickSize"),
        "min_price": price_filter.get("minPrice"),
        "max_price": price_filter.get("maxPrice"),
        "qty_step": lot_filter.get("qtyStep"),
        "min_order_qty": lot_filter.get("minOrderQty"),
        "max_order_qty": lot_filter.get("maxOrderQty"),
        "min_notional_value": lot_filter.get("minNotionalValue"),
        "launch_time": item.get("launchTime"),
        "delivery_time": item.get("deliveryTime"),
    }

def main():
    items, raw_pages = fetch_all_linear_instruments()
    normalized = [normalize(x) for x in items]

    ts = int(time.time())
    raw_path = OUT_DIR / f"bybit_linear_raw.json"
    csv_path = OUT_DIR / f"bybit_linear_normalized.csv"

    with raw_path.open("w", encoding="utf-8") as f:
        json.dump(raw_pages, f, ensure_ascii=False, indent=2)

    fieldnames = list(normalized[0].keys()) if normalized else []
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(normalized)

    print(f"saved raw: {raw_path}")
    print(f"saved csv: {csv_path}")
    print(f"total instruments: {len(normalized)}")

if __name__ == "__main__":
    main()

page=1 items=737 total=737 cursor=no
saved raw: output/bybit/bybit_linear_raw.json
saved csv: output/bybit/bybit_linear_normalized.csv
total instruments: 737


In [8]:
# fetch_okx_swap_instruments.py
import csv
import json
import time
from pathlib import Path

import requests

BASE_URL = "https://www.okx.com"
ENDPOINT = "/api/v5/public/instruments"
OUT_DIR = Path("output/okx")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def fetch_swap_instruments():
    params = {"instType": "SWAP"}
    resp = requests.get(BASE_URL + ENDPOINT, params=params, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    if data.get("code") != "0":
        raise RuntimeError(f"OKX API error: {data}")

    return data.get("data", []), data

def normalize(item):
    inst_id = item.get("instId", "")
    base = item.get("baseCcy")
    quote = item.get("quoteCcy")

    if not base and inst_id.endswith("-SWAP"):
        parts = inst_id.split("-")
        if len(parts) >= 3:
            base = parts[0]
            quote = parts[1]

    return {
        "exchange": "okx",
        "symbol_raw": inst_id,
        "symbol_norm": base,
        "inst_type": item.get("instType"),
        "inst_family": item.get("instFamily"),
        "uly": item.get("uly"),
        "state": item.get("state"),
        "base_coin": base,
        "quote_coin": quote,
        "settle_ccy": item.get("settleCcy"),
        "ct_type": item.get("ctType"),
        "ct_val": item.get("ctVal"),
        "ct_mult": item.get("ctMult"),
        "tick_size": item.get("tickSz"),
        "lot_size": item.get("lotSz"),
        "min_size": item.get("minSz"),
        "list_time": item.get("listTime"),
        "exp_time": item.get("expTime"),
    }

def main():
    items, raw_data = fetch_swap_instruments()
    normalized = [normalize(x) for x in items]

    ts = int(time.time())
    raw_path = OUT_DIR / f"okx_swap_raw.json"
    csv_path = OUT_DIR / f"okx_swap_normalized.csv"

    with raw_path.open("w", encoding="utf-8") as f:
        json.dump(raw_data, f, ensure_ascii=False, indent=2)

    fieldnames = list(normalized[0].keys()) if normalized else []
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(normalized)

    print(f"saved raw: {raw_path}")
    print(f"saved csv: {csv_path}")
    print(f"total instruments: {len(normalized)}")

if __name__ == "__main__":
    main()

saved raw: output/okx/okx_swap_raw.json
saved csv: output/okx/okx_swap_normalized.csv
total instruments: 423


In [16]:
import pandas as pd

# Пути поправь под свои файлы
okx_path = "/Users/mishatrubik/Desktop/spread/output/okx/okx_swap_normalized.csv"
bybit_path = "/Users/mishatrubik/Desktop/spread/output/bybit/bybit_linear_normalized.csv"

okx = pd.read_csv(okx_path)
bybit = pd.read_csv(bybit_path)

# --- 1. Нормализация строковых полей ---
for df in (okx, bybit):
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.strip()

# --- 2. Фильтры для корректного universe ---
okx_f = okx[
    (okx["exchange"] == "okx") &
    (okx["inst_type"] == "SWAP") &
    (okx["state"].str.lower() == "live") &
    (okx["settle_ccy"].str.upper() == "USDT")
].copy()

bybit_f = bybit[
    (bybit["exchange"] == "bybit") &
    (bybit["category"].str.lower() == "linear") &
    (bybit["status"].str.lower() == "trading") &
    (bybit["quote_coin"].str.upper() == "USDT") &
    (bybit["settle_coin"].str.upper() == "USDT")
].copy()

# --- 3. Оставляем только по одной записи на symbol_norm ---
# На случай дублей берём первую; позже можно сделать логику выбора лучшего инструмента
okx_f = okx_f.sort_values(["symbol_norm", "symbol_raw"]).drop_duplicates("symbol_norm", keep="first")
bybit_f = bybit_f.sort_values(["symbol_norm", "symbol_raw"]).drop_duplicates("symbol_norm", keep="first")

# --- 4. Пересечение ---
intersection = pd.merge(
    okx_f,
    bybit_f,
    on="symbol_norm",
    how="inner",
    suffixes=("_okx", "_bybit")
)

# --- 5. Итоговая компактная таблица universe ---
universe = intersection[[
    "symbol_norm",
    "symbol_raw_okx",
    "symbol_raw_bybit",
    "tick_size_okx",
    "lot_size",
    "min_size",
    "tick_size_bybit",
    "qty_step",
    "min_order_qty",
    "min_notional_value"
]].copy()

universe = universe.rename(columns={
    "symbol_norm": "base_coin",
    "symbol_raw_okx": "okx_symbol",
    "symbol_raw_bybit": "bybit_symbol",
    "tick_size_okx": "okx_tick_size",
    "lot_size": "okx_lot_size",
    "min_size": "okx_min_size",
    "tick_size_bybit": "bybit_tick_size",
    "qty_step": "bybit_qty_step",
    "min_order_qty": "bybit_min_order_qty",
    "min_notional_value": "bybit_min_notional_value",
})

universe = universe.sort_values("base_coin").reset_index(drop=True)

# --- 6. Списки тикеров для подписок ---
okx_symbols = universe["okx_symbol"].tolist()
bybit_symbols = universe["bybit_symbol"].tolist()

from pathlib import Path
from datetime import datetime

out_dir = Path("output")
out_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
universe_path = out_dir / f"bybit_okx_universe.csv"
okx_list_path = out_dir / f"okx_symbols.txt"
bybit_list_path = out_dir / f"bybit_symbols.txt"

universe.to_csv(universe_path, index=False)

with open(okx_list_path, "w", encoding="utf-8") as f:
    for s in okx_symbols:
        f.write(f"{s}\n")

with open(bybit_list_path, "w", encoding="utf-8") as f:
    for s in bybit_symbols:
        f.write(f"{s}\n")

print(f"saved universe: {universe_path}")
print(f"saved okx symbols: {okx_list_path}")
print(f"saved bybit symbols: {bybit_list_path}")
print(f"OKX filtered: {len(okx_f)}")
print(f"Bybit filtered: {len(bybit_f)}")
print(f"Intersection size: {len(universe)}")

display(universe.head(30))

print("OKX symbols sample:", okx_symbols[:20])
print("Bybit symbols sample:", bybit_symbols[:20])

saved universe: output/bybit_okx_universe.csv
saved okx symbols: output/okx_symbols.txt
saved bybit symbols: output/bybit_symbols.txt
OKX filtered: 408
Bybit filtered: 633
Intersection size: 348


,base_coin,okx_symbol,bybit_symbol,okx_tick_size,okx_lot_size,okx_min_size,bybit_tick_size,bybit_qty_step,bybit_min_order_qty,bybit_min_notional_value
0,0G,0G-USDT-SWAP,0GUSDT,0.000100,1.00,1.00,0.000100,0.10,0.10,5
1,1INCH,1INCH-USDT-SWAP,1INCHUSDT,0.000010,1.00,1.00,0.000010,0.10,0.10,5
2,2Z,2Z-USDT-SWAP,2ZUSDT,0.000010,1.00,1.00,0.000010,1.00,1.00,5
3,A,A-USDT-SWAP,AUSDT,0.000010,1.00,1.00,0.000010,1.00,1.00,5
4,AAOI,AAOI-USDT-SWAP,AAOIUSDT,0.010000,0.01,0.01,0.010000,0.01,0.01,5
5,AAPL,AAPL-USDT-SWAP,AAPLUSDT,0.010000,0.01,0.01,0.010000,0.01,0.01,5
6,AAVE,AAVE-USDT-SWAP,AAVEUSDT,0.010000,0.10,0.10,0.010000,0.01,0.01,5
7,ACH,ACH-USDT-SWAP,ACHUSDT,0.000001,1.00,1.00,0.000001,10.00,10.00,5
8,ACT,ACT-USDT-SWAP,ACTUSDT,0.000001,1.00,1.00,0.000001,1.00,1.00,5
9,ACU,ACU-USDT-SWAP,ACUUSDT,0.000010,1.00,1.00,0.000010,1.00,1.00,5


OKX symbols sample: ['0G-USDT-SWAP', '1INCH-USDT-SWAP', '2Z-USDT-SWAP', 'A-USDT-SWAP', 'AAOI-USDT-SWAP', 'AAPL-USDT-SWAP', 'AAVE-USDT-SWAP', 'ACH-USDT-SWAP', 'ACT-USDT-SWAP', 'ACU-USDT-SWAP', 'ADA-USDT-SWAP', 'ADBE-USDT-SWAP', 'AERO-USDT-SWAP', 'AEVO-USDT-SWAP', 'AGLD-USDT-SWAP', 'AIXBT-USDT-SWAP', 'ALAB-USDT-SWAP', 'ALGO-USDT-SWAP', 'ALLO-USDT-SWAP', 'AMAT-USDT-SWAP']
Bybit symbols sample: ['0GUSDT', '1INCHUSDT', '2ZUSDT', 'AUSDT', 'AAOIUSDT', 'AAPLUSDT', 'AAVEUSDT', 'ACHUSDT', 'ACTUSDT', 'ACUUSDT', 'ADAUSDT', 'ADBEUSDT', 'AEROUSDT', 'AEVOUSDT', 'AGLDUSDT', 'AIXBTUSDT', 'ALABUSDT', 'ALGOUSDT', 'ALLOUSDT', 'AMATUSDT']
